In [ ]:
!pip install transformers datasets peft accelerate evaluate

In [ ]:
import time
import torch
import numpy as np
import random
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from peft import LoraConfig, get_peft_model
import evaluate

# SETTINGS
MODEL_NAME = "bert-base-uncased"
EPOCHS = 1
BATCH_SIZE = 16
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Using device:", DEVICE)

# LOAD DATA
dataset = load_dataset("glue", "sst2")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(example):
    return tokenizer(
        example["sentence"],
        truncation=True,
        padding="max_length",
        max_length=64
    )

dataset = dataset.map(tokenize, batched=True)
dataset = dataset.rename_column("label", "labels")
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

train_dataset = dataset["train"].select(range(1000))
val_dataset = dataset["validation"].select(range(200))

print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))

# METRIC
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return metric.compute(predictions=preds, references=labels)

# TRAIN FUNCTION (WITH SEED)
def train_model(model, name, seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    model.to(DEVICE)

    training_args = TrainingArguments(
        output_dir=f"./results_{name}",
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=EPOCHS,
        logging_steps=200,
        save_strategy="no",
        report_to="none",
        dataloader_num_workers=2
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics
    )

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    start = time.time()
    trainer.train()
    end = time.time()

    results = trainer.evaluate()

    if torch.cuda.is_available():
        memory = torch.cuda.max_memory_allocated() / 1e9
    else:
        memory = 0

    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6

    return {
        "accuracy": results["eval_accuracy"] * 100,
        "params": trainable_params,
        "time": (end - start),
        "memory": memory
    }

# MULTI-RUN WRAPPER
def run_multiple(model_fn, name, seeds=[1,2,3]):
    results = []

    for s in seeds:
        print(f"{name} | Seed {s}")
        model = model_fn()
        res = train_model(model, name, seed=s)
        results.append(res)

    avg_acc = np.mean([r["accuracy"] for r in results])
    std_acc = np.std([r["accuracy"] for r in results])

    avg_time = np.mean([r["time"] for r in results])
    avg_mem = np.mean([r["memory"] for r in results])
    avg_params = results[0]["params"]

    return {
        "name": name,
        "accuracy": avg_acc,
        "std": std_acc,
        "params": avg_params,
        "time": avg_time,
        "memory": avg_mem
    }

# RUN EXPERIMENTS
results_all = []

# FULL FT
print("\nRunning Full Fine-Tuning...")
results_all.append(run_multiple(
    lambda: AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2),
    "Full FT"
))

# LoRA
for r in [4, 8, 16, 32]:
    print(f"\nRunning LoRA r={r}...")

    def create_model():
        model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

        lora_config = LoraConfig(
            r=r,
            lora_alpha=16,
            target_modules=["query", "value"],
            lora_dropout=0.1,
            bias="none"
        )

        return get_peft_model(model, lora_config)

    results_all.append(run_multiple(create_model, f"LoRA r={r}"))

# PRINT RESULTS
print("\nFINAL RESULTS:\n")

for r in results_all:
    print(f"{r['name']:12} | Acc: {r['accuracy']:.2f} ± {r['std']:.2f} | Params: {r['params']:.2f}M | Time: {r['time']:.2f}s | Mem: {r['memory']:.2f}GB")

print("\nDone.")